# FAISS

**Facebook AI Similarity Search** — a C++/CUDA library (with Python bindings) for fast nearest-neighbor search and clustering over dense vectors.

**Domain:** LLM Inference, Training & Optimization  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**What:** FAISS is a library for **similarity search over dense vectors**. Given a set of vectors (embeddings) and a query vector, it returns the *k* nearest neighbors under L2 distance or inner product, fast, at scales from thousands to billions of vectors. It also ships k-means, PCA, and product quantization.

**The problem it solves:** A brute-force scan (`numpy` argsort over every distance) is exact but O(N·d) per query — fine for 10k vectors, hopeless for 10M. FAISS gives you (a) heavily optimized exact search (SIMD, BLAS, GPU), and (b) a menu of **approximate** indexes that trade a sliver of recall for 10–100× speed and large memory savings via compression.

**Reach for it when:**
- You're building **retrieval** (RAG, semantic search, dedup, recommendation) and the vector count outgrows a naive scan.
- You want an **in-process, dependency-light** ANN engine — no server, no network — embeddable in a script or service.
- You need **fine control** over the speed / memory / recall trade-off (quantization, IVF, HNSW) and the math behind it.

**Don't reach for it when:**
- You need a managed service with metadata filtering, persistence, replication, and CRUD out of the box — use a vector DB (Pinecone, Weaviate, Qdrant, pgvector). Many of them use FAISS or similar under the hood. See [[vector-db-comparison]].
- Your corpus is small (< ~50k) — a flat `numpy`/`sklearn` search is exact, simpler, and plenty fast.

## 2. Mental Model

Think of FAISS as a **library of indexes**, not one algorithm. You pick a recipe along three axes:

1. **Exact vs approximate.** `IndexFlat*` scans everything (100% recall, slow at scale). Everything else is approximate — it skips most of the data.
2. **How it prunes.** `IVF` (inverted file) clusters vectors into Voronoi cells and only searches the few cells near the query. `HNSW` builds a navigable small-world graph and greedily walks toward the query.
3. **How it stores vectors.** Keep them raw (`Flat`), or compress with **Product Quantization** (`PQ`) / scalar quantization to shrink memory 8–32× — searching the compressed codes directly.

The cleanest mental picture for IVF:

```
       query q
          |
   [ coarse quantizer ]   -> pick the nprobe nearest of nlist cluster centroids
          |
   cell 7   cell 42   cell 91     (only these are scanned, not all N vectors)
   [....]   [....]    [....]
          |
   k nearest among the scanned vectors
```

`nprobe` is the knob: probe 1 cell → fast, lower recall; probe more cells → slower, higher recall, approaching exact. Training learns the centroids; adding assigns each vector to a cell.

## 3. Key Concepts

- **Index** — the searchable structure. Built by `faiss.index_factory(d, "...")` or a concrete class like `IndexFlatL2(d)`. All vectors are `float32`, shape `(n, d)`, **C-contiguous**.
- **Metric** — `METRIC_L2` (squared Euclidean; returned distances are *squared*) or `METRIC_INNER_PRODUCT`. For **cosine similarity**, L2-normalize vectors (`faiss.normalize_L2`) and use inner product.
- **Flat** — store raw vectors, exhaustive search. Exact, the recall baseline.
- **IVF (Inverted File)** — partition space into `nlist` Voronoi cells via k-means; at query time scan the `nprobe` closest cells. Must be **trained** before adding.
- **PQ (Product Quantization)** — split each vector into m sub-vectors, quantize each to a small codebook. Massive compression; distances computed on codes. Often combined: `IVF…,PQ`.
- **HNSW** — graph index, excellent recall/latency, no training, but high build time and memory. Good default when RAM isn't the constraint.
- **train / add / search** — the lifecycle. `train(x)` learns centroids/codebooks (no-op for Flat). `add(x)` inserts vectors. `search(xq, k)` returns `(D, I)`: distances and integer IDs, both shape `(nq, k)`; missing slots are `-1`.
- **recall@k** — fraction of true top-k neighbors the approximate index actually returns. The number you tune `nprobe` / `efSearch` against.

## 4. Setup

Install the CPU build (the GPU build, `faiss-gpu`, needs CUDA and is distributed mainly via conda):

```bash
pip install faiss-cpu numpy
# GPU (conda): conda install -c pytorch -c nvidia faiss-gpu
```

FAISS only does the vector math — it has **no opinion** about where embeddings come from. In a real pipeline you'd generate them with a model (e.g. `sentence-transformers`, an OpenAI/Cohere embedding endpoint) and feed the resulting `float32` arrays in. Here we use random/clustered arrays so everything runs on CPU in seconds.

In [1]:
import numpy as np
import faiss

print("faiss", faiss.__version__)
print("numpy", np.__version__)

# Reproducible toy data. In practice these rows are sentence/image embeddings.
rng = np.random.default_rng(42)
d = 64          # embedding dimension
n = 5000        # database size
nq = 5          # number of queries

xb = rng.random((n, d), dtype=np.float32)   # database vectors
xq = rng.random((nq, d), dtype=np.float32)  # query vectors
print("database:", xb.shape, xb.dtype, "| queries:", xq.shape)

faiss 1.14.3
numpy 2.5.0
database: (5000, 64) float32 | queries: (5, 64)


## 5. Worked Examples

### Example 1 — Exact search with `IndexFlatL2`

The baseline: store every vector, scan all of them. 100% recall, and the yardstick every approximate index is measured against. Note the lifecycle — `IndexFlatL2` needs no training, so we go straight to `add` then `search`.

In [2]:
index = faiss.IndexFlatL2(d)        # exact, squared-L2 distance
print("is_trained:", index.is_trained, "| ntotal:", index.ntotal)

index.add(xb)                       # ingest the database
print("ntotal after add:", index.ntotal)

k = 4
D, I = index.search(xq, k)          # D: squared distances, I: neighbor ids
print("\nneighbor ids (I):\n", I)
print("\ndistances (D), query 0:", D[0].round(3))

# Sanity check: FAISS exact search must match a brute-force numpy scan.
d2 = ((xb - xq[0]) ** 2).sum(axis=1)
print("numpy top-4 for query 0:", np.argsort(d2)[:4])
print("faiss top-4 for query 0:", I[0])
assert np.array_equal(np.argsort(d2)[:k], I[0]), "exact search should match numpy"
print("\nExact search matches brute force. ✅")

is_trained: True | ntotal: 0
ntotal after add: 5000

neighbor ids (I):
 [[3473 3982 4978 2615]
 [ 665  963 1116 1793]
 [3669  758 4399 4553]
 [2203 4957  922 2416]
 [ 130 4241 3129 4336]]

distances (D), query 0: [5.873 6.208 6.274 6.326]
numpy top-4 for query 0: [3473 3982 4978 2615]
faiss top-4 for query 0: [3473 3982 4978 2615]

Exact search matches brute force. ✅


### Example 2 — Approximate search with `IndexIVFFlat`, and the `nprobe` knob

IVF clusters the database into `nlist` cells and, per query, scans only the `nprobe` nearest cells. That's the speed win — and the recall cost. Here we build clustered data (closer to how real embeddings look), compute exact ground truth with a flat index, then watch **recall@10 climb as we raise `nprobe`** toward exhaustive search.

In [3]:
# Clustered data: 50 Gaussian blobs — embeddings have this kind of structure.
d = 64; n = 20000; nq = 100
rng = np.random.default_rng(0)
centers = rng.random((50, d), dtype=np.float32)
lab = rng.integers(0, 50, size=n)
xb = (centers[lab] + 0.05 * rng.standard_normal((n, d))).astype(np.float32)
qlab = rng.integers(0, 50, size=nq)
xq = (centers[qlab] + 0.05 * rng.standard_normal((nq, d))).astype(np.float32)

# Ground truth: exact top-10 from a flat index.
flat = faiss.IndexFlatL2(d); flat.add(xb)
_, gt = flat.search(xq, 10)

# IVF index: a coarse quantizer (flat) partitions into nlist cells.
nlist = 100
ivf = faiss.IndexIVFFlat(faiss.IndexFlatL2(d), d, nlist)
ivf.train(xb)        # k-means to learn the nlist centroids — REQUIRED before add
ivf.add(xb)

def recall_at_10(approx, truth):
    hits = sum(len(set(a) & set(t)) for a, t in zip(approx, truth))
    return hits / (truth.shape[0] * truth.shape[1])

for nprobe in (1, 5, 10, 50):
    ivf.nprobe = nprobe
    _, I = ivf.search(xq, 10)
    print(f"nprobe={nprobe:3d}  cells scanned={nprobe:3d}/{nlist}  recall@10={recall_at_10(I, gt):.3f}")

nprobe=  1  cells scanned=  1/100  recall@10=0.716
nprobe=  5  cells scanned=  5/100  recall@10=1.000
nprobe= 10  cells scanned= 10/100  recall@10=1.000
nprobe= 50  cells scanned= 50/100  recall@10=1.000


### Example 3 — Cosine similarity, compression with PQ, and the `index_factory`

Two common needs in one cell:

- **Cosine similarity:** FAISS has no cosine metric. L2-normalize the vectors and use **inner product** (`IndexFlatIP`) — on unit vectors, inner product *is* cosine.
- **Compression:** `index_factory` builds composite indexes from a string. `"IVF100,PQ8"` = IVF with 100 cells, vectors compressed to 8 bytes each via product quantization — ~32× smaller than raw 64-dim `float32` (256 bytes), at some recall cost.

In [4]:
# --- Cosine similarity via normalized inner product ---
xb_n = xb.copy(); faiss.normalize_L2(xb_n)   # in-place; vectors become unit length
xq_n = xq.copy(); faiss.normalize_L2(xq_n)
ip = faiss.IndexFlatIP(d); ip.add(xb_n)
D, I = ip.search(xq_n, 3)
print("cosine sims for query 0:", D[0].round(3), "(1.0 == identical direction)")

# --- Compressed index built from a factory string ---
pq = faiss.index_factory(d, "IVF100,PQ8")    # IVF coarse + 8-byte PQ codes
pq.train(xb)
pq.add(xb)
pq.nprobe = 10
_, Ipq = pq.search(xq, 10)
print("PQ recall@10:", round(recall_at_10(Ipq, gt), 3),
      "| bytes/vector:", 8, "vs raw", d * 4)

# --- Persist and reload (FAISS indexes are just files) ---
faiss.write_index(ip, "/tmp/demo.index")
reloaded = faiss.read_index("/tmp/demo.index")
print("reloaded ntotal:", reloaded.ntotal)

cosine sims for query 0: [0.996 0.996 0.996] (1.0 == identical direction)


PQ recall@10: 0.384 | bytes/vector: 8 vs raw 256
reloaded ntotal: 20000


### Optional — real embeddings (gated, no network unless a model is present)

The cells above use synthetic vectors so they always run. In production you'd embed text first. This cell only runs if `sentence-transformers` is installed (it downloads a small model on first use), so the notebook still executes top-to-bottom without it.

In [5]:
import importlib.util, os

if importlib.util.find_spec("sentence_transformers") and os.getenv("RUN_EMBEDDINGS"):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")   # ~80 MB download
    docs = ["the cat sat on the mat", "dogs are loyal companions",
            "FAISS does fast vector search", "approximate nearest neighbors"]
    emb = model.encode(docs, normalize_embeddings=True).astype("float32")
    idx = faiss.IndexFlatIP(emb.shape[1]); idx.add(emb)
    q = model.encode(["which library searches vectors?"],
                     normalize_embeddings=True).astype("float32")
    D, I = idx.search(q, 2)
    print("top matches:", [docs[i] for i in I[0]], "sims:", D[0].round(3))
else:
    print("Skipped: set RUN_EMBEDDINGS=1 and `pip install sentence-transformers` to run.")
    print("Shape of the call: model.encode(docs) -> float32 (n, d) -> index.add(...).")

Skipped: set RUN_EMBEDDINGS=1 and `pip install sentence-transformers` to run.
Shape of the call: model.encode(docs) -> float32 (n, d) -> index.add(...).


## 6. Gotchas & Pitfalls

- **dtype and contiguity.** FAISS expects `float32`, C-contiguous arrays. `float64` (numpy's default) silently misbehaves or errors — always `.astype('float32')`. After slicing/transposing, `np.ascontiguousarray(x)`.
- **L2 distances are *squared*.** `IndexFlatL2` returns squared Euclidean distance. Take `np.sqrt` if you need true distance; for ranking it doesn't matter.
- **No cosine metric.** Normalize vectors and use `IndexFlatIP` / inner product. Forgetting to normalize gives you raw dot products, not cosine — silently wrong rankings.
- **Forgetting to train.** IVF/PQ indexes raise if you `add` before `train`. Flat and HNSW need no training. Train on a representative sample (you don't need the full set, but ~30–256× `nlist` points is a good rule for IVF).
- **`nlist` / `nprobe` tuning.** A common starting point: `nlist ≈ sqrt(N)`. Too few training points per cell → bad centroids. `nprobe` is your recall dial at query time — measure recall against a flat ground truth, don't guess.
- **IDs are positional by default.** `search` returns row indices into insertion order, not your DB keys. Use `IndexIDMap` / `add_with_ids` to attach your own 64-bit IDs. `-1` in results means "fewer than k neighbors found."
- **No deletes / metadata filtering.** Plain FAISS indexes don't support easy per-vector deletion or attribute filtering. If you need CRUD, soft-deletes, or `WHERE category='x'`, you want a vector DB layer on top.
- **Threading.** FAISS parallelizes a *batch* of queries across threads (OpenMP). Searching one query at a time in a Python loop leaves cores idle — batch your queries.

## 7. When to Use vs Alternatives

| Option | What it is | Use it when | Watch out for |
|---|---|---|---|
| **FAISS (Flat)** | Exact brute force | Corpus small-ish (< ~1M), recall must be 100% | O(N) per query; RAM = full vectors |
| **FAISS (IVF/PQ/HNSW)** | In-process ANN library | You want speed/memory control, no server, billions-scale | You build persistence/CRUD/filtering yourself |
| **HNSWlib / Annoy / ScaNN** | Other ANN libraries | HNSWlib: simple great-recall graph; Annoy: mmap, static; ScaNN: top-tier on benchmarks | Fewer index types than FAISS; Annoy is build-once |
| **pgvector** | Postgres extension | Vectors live next to relational data; you want SQL + transactions | Slower at very large scale than dedicated ANN |
| **Qdrant / Weaviate / Milvus** | Vector databases (self-host) | Need filtering, payloads, CRUD, replication, REST/gRPC | Operational overhead; many wrap FAISS-like cores |
| **Pinecone** | Managed vector DB (SaaS) | Don't want to run infra; need scaling + filtering | Cost; data leaves your box |
| **numpy / sklearn** | Plain array math | A few thousand vectors, prototyping | Doesn't scale; no compression |

**Rule of thumb:** FAISS is the *engine*, not the *database*. If you just need fast k-NN inside a process and will manage the rest yourself, FAISS is the sharpest tool. The moment you need filtering, persistence-as-a-service, multi-tenancy, and CRUD, put a vector DB in front (often FAISS-powered anyway). See [[vector-db-comparison]] and [[vector-embeddings]] for the surrounding pipeline.

## 8. Resources

- **Official wiki (start here):** https://github.com/facebookresearch/faiss/wiki — especially *Guidelines to choose an index* and *Faster search*.
- **GitHub repo:** https://github.com/facebookresearch/faiss
- **"Guidelines to choose an index":** https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index — the decision tree for Flat vs IVF vs PQ vs HNSW by dataset size and RAM.
- **The IVF/PQ paper** (Jégou et al., *Product Quantization for Nearest Neighbor Search*): https://hal.inria.fr/inria-00514462/document
- **Pinecone's FAISS guide** (clear, example-driven walkthrough): https://www.pinecone.io/learn/series/faiss/
- **ann-benchmarks** (how FAISS indexes compare to other ANN libs): https://ann-benchmarks.com/